# CS 521 HW3 — Question 2





In [ ]:
import subprocess
import sys

if sys.version_info >= (3, 13):
    raise RuntimeError("SynCode 0.4.16 requires Python 3.12 or earlier.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "syncode==0.4.16", "transformers==4.53.2", "accelerate>=0.26.0"],
    check=True,
)

import json
import math
import operator
import torch
from syncode import Grammar, SyncodeLogitsProcessor
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
)

MODEL_ID = "microsoft/phi-2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Phi-2 can overflow in float16. Use bfloat16 on supported GPUs and float32 otherwise.
DTYPE = (
    torch.bfloat16
    if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
if DEVICE.type == "cpu":
    print("No GPU detected. Phi-2 will run slowly and needs substantial RAM.")

In [ ]:
# This grammar permits exactly two numbers and one of the four binary operators. The LLM extracts these fields; the grammar does not perform arithmetic.
EXTRACTION_GRAMMAR = r'''
start: "{" "\"operands\"" ":" "[" NUMBER "," NUMBER "]" "," "\"operator\"" ":" OP "}"
NUMBER: /-?(0|[1-9][0-9]*)(\.[0-9]+)?([eE][+-]?[0-9]+)?/
OP: "\"+\"" | "\"-\"" | "\"*\"" | "\"/\""
%ignore /[ \t\r\n]+/
'''

In [ ]:
# I add 4 few-shot examples here
FEW_SHOT_PREFIX = '''Instruct: Extract the two numeric operands and the single binary operator from the query.
Return only a JSON object in this exact format:
{"operands": [first_operand, second_operand], "operator": "+"}
The operator must be one of +, -, *, /.
Copy numeric values accurately and express them as numbers, preferably decimals.
Put operands in calculation order: "subtract A from B" means [B, A] with operator "-".
"A divided by B" means [A, B] with operator "/".
Extract only. Do not calculate or include a final answer or explanation.

Query: What is 12.5 plus 3.25?
Extraction: {"operands": [12.5, 3.25], "operator": "+"}

Query: Subtract 2.0 from 9.5.
Extraction: {"operands": [9.5, 2.0], "operator": "-"}

Query: What is 7. multiplied by -2.0?
Extraction: {"operands": [7.0, -2.0], "operator": "*"}

Query: What is 18.6 divided by 3.0?
Extraction: {"operands": [18.6, 3.0], "operator": "/"}

'''

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": str(DEVICE)},
    low_cpu_mem_usage=True,
    attn_implementation="eager",
).eval()

syncode_processor = SyncodeLogitsProcessor(
    grammar=Grammar(EXTRACTION_GRAMMAR),
    tokenizer=tokenizer,
    parse_output_only=True,
    mode="grammar_strict",
    num_samples=1,
)

print(f"Ready: {MODEL_ID}, device={DEVICE}, dtype={DTYPE}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Ready: microsoft/phi-2, device=cuda, dtype=torch.bfloat16


In [ ]:
from syncode import Grammar, SyncodeLogitsProcessor

# Separate number components so SynCode permits decimal-point tokens. I had to do this after facing decimal based errors, the numbers would just round off and that is not what we need so we got errors
EXTRACTION_GRAMMAR = r'''
start: BEGIN_OBJECT OPERANDS COLON BEGIN_ARRAY number COMMA number END_ARRAY COMMA OPERATOR COLON OP END_OBJECT

number: MINUS? INT FRAC? EXP?
MINUS: "-"
INT: /0|[1-9][0-9]*/
FRAC: /\.[0-9]+/
EXP: /[eE][+-]?[0-9]+/

BEGIN_OBJECT: /[ \t\r\n]*\{[ \t\r\n]*/
END_OBJECT: /[ \t\r\n]*\}[ \t\r\n]*/
BEGIN_ARRAY: /[ \t\r\n]*\[[ \t\r\n]*/
END_ARRAY: /[ \t\r\n]*\][ \t\r\n]*/
COMMA: /[ \t\r\n]*,[ \t\r\n]*/
COLON: /[ \t\r\n]*:[ \t\r\n]*/

OPERANDS: "\"operands\""
OPERATOR: "\"operator\""
OP: "\"+\"" | "\"-\"" | "\"*\"" | "\"/\""
'''

syncode_processor = SyncodeLogitsProcessor(
    grammar=Grammar(EXTRACTION_GRAMMAR),
    tokenizer=tokenizer,
    parse_output_only=True,
    mode="grammar_strict",
    num_samples=1,
)

In [ ]:
class _StopAfterExtraction(StoppingCriteria):

    def __init__(self, prompt_length):
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        completed = [
            tokenizer.decode(
                row[self.prompt_length:], skip_special_tokens=True
            ).rstrip().endswith("}")
            for row in input_ids
        ]
        return torch.tensor(completed, dtype=torch.bool, device=input_ids.device)


In [ ]:
def _parse_extraction(raw_output):
  try:
        data = json.loads(raw_output)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Phi-2 returned invalid JSON: {raw_output!r}") from exc

    if not isinstance(data, dict) or set(data) != {"operands", "operator"}:
        raise ValueError(f"Unexpected extraction fields: {data!r}")
    operands = data["operands"]
    op = data["operator"]
    if not isinstance(operands, list) or len(operands) != 2:
        raise ValueError("The extraction must contain exactly two operands.")
    if any(type(value) not in (int, float) for value in operands):
        raise ValueError("Both operands must be numbers.")
    operands = [float(value) for value in operands]
    if not all(math.isfinite(value) for value in operands):
        raise ValueError("Both operands must be finite floating-point values.")
    if not isinstance(op, str) or op not in {"+", "-", "*", "/"}:
        raise ValueError("The extracted operator must be +, -, *, or /.")
    return operands, op


_OPERATIONS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}

In [ ]:
def _calculate(operands, op):
    if op == "/" and operands[1] == 0.0:
        raise ZeroDivisionError("The extracted divisor is zero.")
    return float(_OPERATIONS[op](operands[0], operands[1]))

#this is the run_syncode as expected in teh HW file; this is also dependendent on teh functions baove, so try not to isolate it while testing
def run_syncode(textual_query):
    if not isinstance(textual_query, str) or not textual_query.strip():
        raise ValueError("textual_query must be a nonempty string.")

    prompt = FEW_SHOT_PREFIX + "Query: " + textual_query.strip() + "\nExtraction:"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_length = inputs["input_ids"].shape[1]
    max_new_tokens = 128
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("The query is too long for Phi-2's context window.")

    syncode_processor.reset()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            num_return_sequences=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            logits_processor=[syncode_processor],
            stopping_criteria=StoppingCriteriaList([
                _StopAfterExtraction(prompt_length)
            ]),
        )

    raw_output = tokenizer.decode(
        output_ids[0, prompt_length:], skip_special_tokens=True
    ).strip()
    operands, op = _parse_extraction(raw_output)
    final_answer = _calculate(operands, op)
    return operands, op, final_answer

In [10]:
# Actual model tests, I have included the examples in the assignment for the sake of uniformity
test_cases = [
    ("What is 327. multiplied by 11.0?", [327.0, 11.0], "*", 3597.0),
    ("What is 45.1 plus 23.54?", [45.1, 23.54], "+", 68.64),
    ("What is 120.4 divided by 4.0?", [120.4, 4.0], "/", 30.1),
    ("What is 18.75 minus 6.25?", [18.75, 6.25], "-", 12.5),
    ("Subtract 3.25 from 10.5.", [10.5, 3.25], "-", 7.25),
    ("What is -2.5 times 4.0?", [-2.5, 4.0], "*", -10.0),
]

passed = 0
for query, expected_operands, expected_op, expected_answer in test_cases:
    try:
        result = run_syncode(query)
        operands, op, answer = result
        ok = (
            isinstance(result, tuple) and len(result) == 3
            and isinstance(operands, list) and len(operands) == 2
            and all(type(value) is float for value in operands)
            and all(math.isclose(a, b, rel_tol=1e-7, abs_tol=1e-7)
                    for a, b in zip(operands, expected_operands))
            and op == expected_op
            and type(answer) is float
            and math.isclose(answer, expected_answer, rel_tol=1e-7, abs_tol=1e-7)
        )
        passed += int(ok)
        print(f"{'PASS' if ok else 'FAIL'} | {query}\n  {result}")
    except Exception as exc:
        print(f"FAIL | {query}\n  {type(exc).__name__}: {exc}")

print(f"\nModel tests passed: {passed}/{len(test_cases)}")


PASS | What is 327. multiplied by 11.0?
  ([327.0, 11.0], '*', 3597.0)
PASS | What is 45.1 plus 23.54?
  ([45.1, 23.54], '+', 68.64)
PASS | What is 120.4 divided by 4.0?
  ([120.4, 4.0], '/', 30.1)
PASS | What is 18.75 minus 6.25?
  ([18.75, 6.25], '-', 12.5)
FAIL | Subtract 3.25 from 10.5.
  ValueError: Phi-2 returned incomplete or invalid JSON: '{\n  "operands" : [\n    10.5,\n    3.25\n  ]'
FAIL | What is -2.5 times 4.0?
  ValueError: Phi-2 returned incomplete or invalid JSON: '{\n  "operands" : [\n    -2.5,\n    4.0\n  ]'

Model tests passed: 4/6
